In [28]:
# -*- coding: utf-8 -*-
"""
Make ONE monthly panel that contains:
 - 8 primary emotions (MEAN: P_*, MAX: M_*)
 - polarity mean (polarity_mean)
 - meta columns, review_count, month_index, gvar_month (from first post_netflix==1 per movie)

Output:
  - movie_monthly_panel_emotions_polarity.csv
  - movie_monthly_panel_emotions_polarity.dta
"""

import pandas as pd
import numpy as np

# =========================
# 0) SETTINGS / PATH
# =========================
INPUT_CSV = "movie_long_with_emotions and polarity_full.csv"  # polarity_pm1 포함
STATA_VERSION = 118
META_COLS = ["OTT공개일", "대표국적", "장르"]

# (선택) 달력 범위 고정
FIXED_RANGE_START = None
FIXED_RANGE_END   = None

# =========================
# 1) Emotion column mapping
# =========================
JOY_COLS = list(set([
    "기쁨_prob","행복_prob","즐거움/신남_prob","흐뭇함(귀여움/예쁨)_prob",
    "흐뭇함_귀여움_예쁨__prob","뿌듯함_prob","감동/감탄_prob","편안/쾌적_prob"
]))
TRUST_COLS = list(set(["안심/신뢰_prob","환영/호의_prob","존경_prob","아껴주는_prob"]))
FEAR_COLS  = list(set(["공포/무서움_prob","불안/걱정_prob","부담/안_내킴_prob"]))
SURPRISE_COLS = list(set(["놀람_prob","경악_prob","당황/난처_prob","깨달음_prob"]))
SADNESS_COLS  = list(set([
    "슬픔_prob","절망_prob","서러움_prob","안타까움/실망_prob","재미없음_prob",
    "힘듦/지침_prob","패배/자기혐오_prob","죄책감_prob","불쌍함/연민_prob"
]))
DISGUST_COLS = list(set(["역겨움/징그러움_prob","한심함_prob","어이없음_prob","지긋지긋_prob"]))
ANGER_COLS   = list(set(["화남/분노_prob","짜증_prob","불평/불만_prob"]))
ANTICIP_COLS = list(set(["기대감_prob","신기함/관심_prob","의심/불신_prob"]))

BLEND_MAP = {
    "증오/혐오_prob": {"DISGUST": 0.5, "ANGER": 0.5},
    "우쭐댐/무시함_prob": {"DISGUST": 0.5, "ANGER": 0.5},
    "고마움_prob": {"JOY": 0.5, "TRUST": 0.5},
    "부끄러움_prob": {"SADNESS": 0.5, "FEAR": 0.5},
}

PRIMARY_EMOTIONS = ["JOY", "TRUST", "FEAR", "SURPRISE", "SADNESS", "DISGUST", "ANGER", "ANTICIP"]
EMOTION_TO_COLS = {
    "JOY": JOY_COLS, "TRUST": TRUST_COLS, "FEAR": FEAR_COLS, "SURPRISE": SURPRISE_COLS,
    "SADNESS": SADNESS_COLS, "DISGUST": DISGUST_COLS, "ANGER": ANGER_COLS, "ANTICIP": ANTICIP_COLS
}

# =========================
# 2) Utilities
# =========================
def read_csv_kor(path: str) -> pd.DataFrame:
    """한국어 CSV 인코딩에 안전하게 로드"""
    for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8", "latin1"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    return pd.read_csv(path)

def to_float_series(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce").astype(float)

def compute_mean_emotions(df: pd.DataFrame) -> pd.DataFrame:
    """가중 평균(혼합 매핑 포함)으로 P_{EMOTION} 열 생성"""
    out = df.copy()
    for emo in PRIMARY_EMOTIONS:
        new_col = f"P_{emo}"
        weighted_cols, weights = [], []

        for col in EMOTION_TO_COLS[emo]:
            if col in out.columns:
                weighted_cols.append(to_float_series(out[col])); weights.append(1.0)

        for blend_col, mapping in BLEND_MAP.items():
            if emo in mapping and blend_col in out.columns:
                weighted_cols.append(to_float_series(out[blend_col])); weights.append(float(mapping[emo]))

        if weighted_cols:
            wsum = np.zeros(len(out), dtype=float)
            for series, w in zip(weighted_cols, weights):
                wsum += series.fillna(0).to_numpy() * w
            out[new_col] = wsum / float(np.sum(weights))
        else:
            out[new_col] = np.nan
    return out

def compute_max_emotions(df: pd.DataFrame) -> pd.DataFrame:
    """최댓값으로 M_{EMOTION} 열 생성 (기존 로직 유지: 혼합 컬럼 전체 후보 포함)"""
    out = df.copy()
    for emo in PRIMARY_EMOTIONS:
        new_col = f"M_{emo}"
        candidates = list(EMOTION_TO_COLS[emo]) + list(BLEND_MAP.keys())
        exist = [c for c in candidates if c in out.columns]
        if exist:
            out[new_col] = out[exist].apply(pd.to_numeric, errors='coerce').max(axis=1)
        else:
            out[new_col] = np.nan
    return out

def create_monthly_panel(dataframe: pd.DataFrame, value_cols: list) -> pd.DataFrame:
    """월별 패널 생성 (value_cols를 월평균으로 집계)"""
    df = dataframe.copy()
    df["month_start"] = df["DATE"].dt.to_period("M").dt.start_time

    has_review_text = ("감상평" in df.columns)
    agg_dict = {col: (col, "mean") for col in value_cols}
    if has_review_text:
        agg_dict["review_count"] = ("감상평", "count")
    for c in META_COLS:
        if c in df.columns:
            agg_dict[c] = (c, "first")

    df_monthly_agg = df.groupby(["영화명", "month_start"]).agg(**agg_dict).reset_index()

    if not has_review_text:
        sz = df.groupby(["영화명", "month_start"]).size().reset_index(name="review_count")
        df_monthly_agg = df_monthly_agg.merge(sz, on=["영화명", "month_start"], how="left")

    # 달력 스켈레톤
    if FIXED_RANGE_START and FIXED_RANGE_END:
        date_range = pd.date_range(start=pd.Timestamp(FIXED_RANGE_START),
                                   end=pd.Timestamp(FIXED_RANGE_END), freq="MS")
    else:
        min_m = df["DATE"].dt.to_period("M").min().to_timestamp(how="start")
        max_m = df["DATE"].dt.to_period("M").max().to_timestamp(how="start")
        date_range = pd.date_range(start=min_m, end=max_m, freq="MS")

    movies = df["영화명"].unique()
    skeleton = pd.MultiIndex.from_product([movies, date_range], names=["영화명", "month_start"])
    panel = pd.DataFrame(index=skeleton).reset_index()

    # 병합
    panel = panel.merge(df_monthly_agg, on=["영화명", "month_start"], how="left")

    # 메타 전파
    for c in META_COLS:
        if c in panel.columns:
            panel[c] = panel.groupby("영화명")[c].transform("ffill").transform("bfill")

    # month_index
    first_month = panel["month_start"].min()
    panel["month_index"] = (
        (panel["month_start"].dt.year - first_month.year) * 12
        + (panel["month_start"].dt.month - first_month.month)
    )

    # gvar_month (영화별 상수 OTT공개일 기준)
    if "OTT공개일" in panel.columns:
        panel["OTT공개일"] = pd.to_datetime(panel["OTT공개일"], errors="coerce")
        gstart = panel["OTT공개일"].dt.to_period("M").dt.start_time
        panel["gvar_month"] = np.where(
            pd.notna(gstart),
            (gstart.dt.year - first_month.year) * 12 + (gstart.dt.month - first_month.month),
            np.nan
        )
    else:
        panel["gvar_month"] = np.nan

    return panel

def coverage_summary(panel: pd.DataFrame, name: str) -> None:
    print(f"\n[{name}] rows={len(panel):,}, movies={panel['영화명'].nunique():,}, "
          f"months={panel['month_start'].nunique():,}")
    if "review_count" in panel.columns:
        print(f"  review_count not-null cells: {panel['review_count'].notna().sum():,}")
    if "gvar_month" in panel.columns:
        print(f"  treated (gvar_month notna) cells: {panel['gvar_month'].notna().sum():,}")

def ensure_gvar(panel: pd.DataFrame) -> pd.DataFrame:
    """최종 병합 후에도 gvar_month가 확실히 들어가도록 보강"""
    if "OTT공개일" not in panel.columns or "month_start" not in panel.columns:
        return panel
    first_month = panel["month_start"].min()
    gstart = pd.to_datetime(panel["OTT공개일"], errors="coerce").dt.to_period("M").dt.start_time
    panel["gvar_month"] = np.where(
        pd.notna(gstart),
        (gstart.dt.year - first_month.year) * 12 + (gstart.dt.month - first_month.month),
        np.nan
    )
    return panel


In [30]:
# =========================
# 3) MAIN
# =========================
def main():
    print("1) Load CSV ...")
    df = read_csv_kor(INPUT_CSV)

    print("2) Parse dates / typing ...")
    # DATE
    df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")
    df = df.dropna(subset=["DATE"]).copy()

    # 감정 원천 숫자형(0~1) 보정
    all_source_cols = set().union(*EMOTION_TO_COLS.values()).union(BLEND_MAP.keys())
    for c in (all_source_cols & set(df.columns)):
        df[c] = pd.to_numeric(df[c], errors="coerce").clip(0, 1)

    # polarity 숫자형 & 범위
    if "polarity_pm1" not in df.columns:
        raise KeyError("입력 CSV에 'polarity_pm1'이 없습니다. 파일을 확인하세요.")
    df["polarity_pm1"] = pd.to_numeric(df["polarity_pm1"], errors="coerce").clip(-1, 1)

    # === 핵심: OTT공개일 = 영화별 post_netflix==1 최초 DATE ===
    if "post_netflix" in df.columns:
        df["post_netflix"] = pd.to_numeric(df["post_netflix"], errors="coerce")
        first_ott = (
            df.loc[df["post_netflix"] == 1, ["영화명", "DATE"]]
              .groupby("영화명", as_index=False)["DATE"].min()
              .rename(columns={"DATE": "OTT공개일_calc"})
        )
        # 기존 OTT공개일이 있더라도 "calc"를 우선 사용 (정의 확정)
        if "OTT공개일" in df.columns:
            df = df.drop(columns=["OTT공개일"])
        df = df.merge(first_ott, on="영화명", how="left")
        df = df.rename(columns={"OTT공개일_calc": "OTT공개일"})
    # (post_netflix 없으면, META_COLS에 OTT공개일이 없을 수 있음 → gvar_month는 NaN 유지)

    # 3) MEAN 패널 (8감정)
    print("3) Build MEAN-based emotion probabilities ...")
    df_mean = compute_mean_emotions(df)
    mean_cols = [f"P_{e}" for e in PRIMARY_EMOTIONS]
    panel_mean = create_monthly_panel(df_mean, mean_cols)
    coverage_summary(panel_mean, "MEAN panel (emotions)")

    # 4) MAX 패널 (8감정) — 기존 로직
    print("4) Build MAX-based emotion probabilities ...")
    df_max = compute_max_emotions(df)
    max_cols = [f"M_{e}" for e in PRIMARY_EMOTIONS]
    panel_max = create_monthly_panel(df_max, max_cols)
    coverage_summary(panel_max, "MAX panel (emotions)")

    # 5) POLARITY 월평균 패널
    print("5) Build monthly polarity mean ...")
    panel_pol = create_monthly_panel(df, ["polarity_pm1"]).rename(columns={"polarity_pm1": "polarity_mean"})
    coverage_summary(panel_pol, "POLARITY mean panel")

    # 6) 하나로 합치기 (영화명, month_start 기준)
    print("6) Merge panels into one ...")
    # mean부터 시작
    merged = panel_mean.copy()

    # max에서 M_* 만 가져와 병합 (메타/카운트는 mean 쪽 유지)
    keep_max = ["영화명", "month_start"] + [c for c in panel_max.columns if c.startswith("M_")]
    merged = merged.merge(panel_max[keep_max], on=["영화명", "month_start"], how="left")

    # polarity_mean 붙이기
    merged = merged.merge(panel_pol[["영화명", "month_start", "polarity_mean"]],
                          on=["영화명", "month_start"], how="left")

    # gvar_month 보강 계산 (혹시 중간에서 빠졌을 경우 대비)
    merged = ensure_gvar(merged)

    # 컬럼 정렬: 키/메타 -> P_* -> M_* -> polarity_mean
    base_cols = ["영화명", "month_start", "review_count", "OTT공개일", "대표국적", "장르",
                 "month_index", "gvar_month"]
    base_cols = [c for c in base_cols if c in merged.columns]
    order = ["영화명", "month_start"] + [c for c in base_cols if c not in ["영화명","month_start"]]
    order += [c for c in merged.columns if c.startswith("P_")]
    order += [c for c in merged.columns if c.startswith("M_")]
    if "polarity_mean" in merged.columns:
        order += ["polarity_mean"]
    order += [c for c in merged.columns if c not in order]
    merged = merged[order]

    # 7) 저장
    out_csv = "movie_monthly_panel_emotions_polarity.csv"
    out_dta = "movie_monthly_panel_emotions_polarity.dta"
    merged.to_csv(out_csv, index=False, encoding="utf-8-sig")
    merged.to_stata(out_dta, write_index=False, version=STATA_VERSION)

    print(f"\n -> Save: {out_csv} / {out_dta}")
    coverage_summary(merged, "FINAL merged panel")

    print("\n✨ All done.")

if __name__ == "__main__":
    main()


1) Load CSV ...
2) Parse dates / typing ...
3) Build MEAN-based emotion probabilities ...

[MEAN panel (emotions)] rows=12,738, movies=193, months=66
  review_count not-null cells: 3,481
  treated (gvar_month notna) cells: 12,738
4) Build MAX-based emotion probabilities ...

[MAX panel (emotions)] rows=12,738, movies=193, months=66
  review_count not-null cells: 3,481
  treated (gvar_month notna) cells: 12,738
5) Build monthly polarity mean ...

[POLARITY mean panel] rows=12,738, movies=193, months=66
  review_count not-null cells: 3,481
  treated (gvar_month notna) cells: 12,738
6) Merge panels into one ...

 -> Save: movie_monthly_panel_emotions_polarity.csv / movie_monthly_panel_emotions_polarity.dta

[FINAL merged panel] rows=12,738, movies=193, months=66
  review_count not-null cells: 3,481
  treated (gvar_month notna) cells: 12,738

✨ All done.


In [32]:
# -*- coding: utf-8 -*-
"""
Build two monthly panels (MEAN / MAX) for 8 emotions + monthly polarity mean,
with meta (OTT공개일 from first post_netflix==1 per movie), review_count,
month_index, and gvar_month.

Outputs:
  - movie_monthly_panel_emotions_mean.csv / .dta   (P_* + polarity_mean)
  - movie_monthly_panel_emotions_max.csv  / .dta   (M_* + polarity_mean)
"""

import pandas as pd
import numpy as np

# =========================
# SETTINGS
# =========================
INPUT_CSV = "movie_long_with_emotions and polarity_full.csv"
STATA_VERSION = 118
META_COLS = ["OTT공개일", "대표국적", "장르"]

# (선택) 달력 범위 고정
FIXED_RANGE_START = None
FIXED_RANGE_END   = None

# =========================
# Emotion mapping
# =========================
JOY_COLS = list(set([
    "기쁨_prob","행복_prob","즐거움/신남_prob","흐뭇함(귀여움/예쁨)_prob",
    "흐뭇함_귀여움_예쁨__prob","뿌듯함_prob","감동/감탄_prob","편안/쾌적_prob"
]))
TRUST_COLS = list(set(["안심/신뢰_prob","환영/호의_prob","존경_prob","아껴주는_prob"]))
FEAR_COLS  = list(set(["공포/무서움_prob","불안/걱정_prob","부담/안_내킴_prob"]))
SURPRISE_COLS = list(set(["놀람_prob","경악_prob","당황/난처_prob","깨달음_prob"]))
SADNESS_COLS  = list(set([
    "슬픔_prob","절망_prob","서러움_prob","안타까움/실망_prob","재미없음_prob",
    "힘듦/지침_prob","패배/자기혐오_prob","죄책감_prob","불쌍함/연민_prob"
]))
DISGUST_COLS = list(set(["역겨움/징그러움_prob","한심함_prob","어이없음_prob","지긋지긋_prob"]))
ANGER_COLS   = list(set(["화남/분노_prob","짜증_prob","불평/불만_prob"]))
ANTICIP_COLS = list(set(["기대감_prob","신기함/관심_prob","의심/불신_prob"]))

BLEND_MAP = {
    "증오/혐오_prob": {"DISGUST": 0.5, "ANGER": 0.5},
    "우쭐댐/무시함_prob": {"DISGUST": 0.5, "ANGER": 0.5},
    "고마움_prob": {"JOY": 0.5, "TRUST": 0.5},
    "부끄러움_prob": {"SADNESS": 0.5, "FEAR": 0.5},
}

PRIMARY_EMOTIONS = ["JOY", "TRUST", "FEAR", "SURPRISE", "SADNESS", "DISGUST", "ANGER", "ANTICIP"]
EMOTION_TO_COLS = {
    "JOY": JOY_COLS, "TRUST": TRUST_COLS, "FEAR": FEAR_COLS, "SURPRISE": SURPRISE_COLS,
    "SADNESS": SADNESS_COLS, "DISGUST": DISGUST_COLS, "ANGER": ANGER_COLS, "ANTICIP": ANTICIP_COLS
}

# =========================
# Utils
# =========================
def read_csv_kor(path: str) -> pd.DataFrame:
    for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8", "latin1"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    return pd.read_csv(path)

def to_float_series(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce").astype(float)

def compute_mean_emotions(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for emo in PRIMARY_EMOTIONS:
        new_col = f"P_{emo}"
        weighted_cols, weights = [], []
        for col in EMOTION_TO_COLS[emo]:
            if col in out.columns:
                weighted_cols.append(to_float_series(out[col])); weights.append(1.0)
        for blend_col, mapping in BLEND_MAP.items():
            if emo in mapping and blend_col in out.columns:
                weighted_cols.append(to_float_series(out[blend_col])); weights.append(float(mapping[emo]))
        if weighted_cols:
            wsum = np.zeros(len(out), dtype=float)
            for series, w in zip(weighted_cols, weights):
                wsum += series.fillna(0).to_numpy() * w
            out[new_col] = wsum / float(np.sum(weights))
        else:
            out[new_col] = np.nan
    return out

def compute_max_emotions(df: pd.DataFrame) -> pd.DataFrame:
    # 기존 로직: 혼합 컬럼 전부 후보 포함 (원하면 FIXED 버전으로 바꿔도 됨)
    out = df.copy()
    for emo in PRIMARY_EMOTIONS:
        new_col = f"M_{emo}"
        candidates = list(EMOTION_TO_COLS[emo]) + list(BLEND_MAP.keys())
        exist = [c for c in candidates if c in out.columns]
        out[new_col] = out[exist].apply(pd.to_numeric, errors='coerce').max(axis=1) if exist else np.nan
    return out

def create_monthly_panel(dataframe: pd.DataFrame, value_cols: list) -> pd.DataFrame:
    """value_cols를 영화×월 평균으로 집계하고 스켈레톤 달력과 메타/지표를 붙임"""
    df = dataframe.copy()
    df["month_start"] = df["DATE"].dt.to_period("M").dt.start_time

    has_review_text = ("감상평" in df.columns)
    agg_dict = {col: (col, "mean") for col in value_cols}
    if has_review_text:
        agg_dict["review_count"] = ("감상평", "count")
    for c in META_COLS:
        if c in df.columns:
            agg_dict[c] = (c, "first")

    df_monthly_agg = df.groupby(["영화명", "month_start"]).agg(**agg_dict).reset_index()

    if not has_review_text:
        sz = df.groupby(["영화명", "month_start"]).size().reset_index(name="review_count")
        df_monthly_agg = df_monthly_agg.merge(sz, on=["영화명", "month_start"], how="left")

    # skeleton calendar
    if FIXED_RANGE_START and FIXED_RANGE_END:
        date_range = pd.date_range(start=pd.Timestamp(FIXED_RANGE_START),
                                   end=pd.Timestamp(FIXED_RANGE_END), freq="MS")
    else:
        min_m = df["DATE"].dt.to_period("M").min().to_timestamp(how="start")
        max_m = df["DATE"].dt.to_period("M").max().to_timestamp(how="start")
        date_range = pd.date_range(start=min_m, end=max_m, freq="MS")

    movies = df["영화명"].unique()
    skeleton = pd.MultiIndex.from_product([movies, date_range], names=["영화명", "month_start"])
    panel = pd.DataFrame(index=skeleton).reset_index()

    panel = panel.merge(df_monthly_agg, on=["영화명", "month_start"], how="left")

    # meta forward/back fill per movie
    for c in META_COLS:
        if c in panel.columns:
            panel[c] = panel.groupby("영화명")[c].transform("ffill").transform("bfill")

    # month_index (global 0-based)
    first_month = panel["month_start"].min()
    panel["month_index"] = (
        (panel["month_start"].dt.year - first_month.year) * 12
        + (panel["month_start"].dt.month - first_month.month)
    )

    # gvar_month: 공개월(월단위) 기준 이벤트 타임 인덱스(영화별 상수)
    if "OTT공개일" in panel.columns:
        panel["OTT공개일"] = pd.to_datetime(panel["OTT공개일"], errors="coerce")
        gstart = panel["OTT공개일"].dt.to_period("M").dt.start_time
        panel["gvar_month"] = np.where(
            pd.notna(gstart),
            (gstart.dt.year - first_month.year) * 12 + (gstart.dt.month - first_month.month),
            np.nan
        )
    else:
        panel["gvar_month"] = np.nan

    return panel

def ensure_gvar(panel: pd.DataFrame) -> pd.DataFrame:
    if "OTT공개일" not in panel.columns or "month_start" not in panel.columns:
        return panel
    first_month = panel["month_start"].min()
    gstart = pd.to_datetime(panel["OTT공개일"], errors="coerce").dt.to_period("M").dt.start_time
    panel["gvar_month"] = np.where(
        pd.notna(gstart),
        (gstart.dt.year - first_month.year) * 12 + (gstart.dt.month - first_month.month),
        np.nan
    )
    return panel

def coverage_summary(panel: pd.DataFrame, name: str) -> None:
    print(f"\n[{name}] rows={len(panel):,}, movies={panel['영화명'].nunique():,}, "
          f"months={panel['month_start'].nunique():,}")
    if "review_count" in panel.columns:
        print(f"  review_count not-null cells: {panel['review_count'].notna().sum():,}")
    if "gvar_month" in panel.columns:
        print(f"  treated (gvar_month notna) cells: {panel['gvar_month'].notna().sum():,}")


In [34]:
# =========================
# MAIN
# =========================
def main():
    print("1) Load CSV ...")
    df = read_csv_kor(INPUT_CSV)

    print("2) Parse & typing ...")
    df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")
    df = df.dropna(subset=["DATE"]).copy()

    # emotion sources to [0,1]
    all_source_cols = set().union(*EMOTION_TO_COLS.values()).union(BLEND_MAP.keys())
    for c in (all_source_cols & set(df.columns)):
        df[c] = pd.to_numeric(df[c], errors="coerce").clip(0, 1)

    # polarity [-1,1]
    if "polarity_pm1" not in df.columns:
        raise KeyError("입력 CSV에 'polarity_pm1'이 없습니다.")
    df["polarity_pm1"] = pd.to_numeric(df["polarity_pm1"], errors="coerce").clip(-1, 1)

    # OTT공개일 = 영화별 post_netflix==1 최초 DATE
    if "post_netflix" in df.columns:
        df["post_netflix"] = pd.to_numeric(df["post_netflix"], errors="coerce")
        first_ott = (
            df.loc[df["post_netflix"] == 1, ["영화명", "DATE"]]
              .groupby("영화명", as_index=False)["DATE"].min()
              .rename(columns={"DATE": "OTT공개일"})
        )
        if "OTT공개일" in df.columns:
            df = df.drop(columns=["OTT공개일"])
        df = df.merge(first_ott, on="영화명", how="left")

    # -------- Build MEAN panel (P_* + polarity_mean) --------
    print("3) Build MEAN (P_*) ...")
    df_mean = compute_mean_emotions(df)
    mean_cols = [f"P_{e}" for e in PRIMARY_EMOTIONS]
    panel_mean = create_monthly_panel(df_mean, mean_cols)

    print("3b) Attach monthly polarity_mean ...")
    pol_panel = create_monthly_panel(df, ["polarity_pm1"]).rename(columns={"polarity_pm1": "polarity_mean"})
    panel_mean = panel_mean.merge(
        pol_panel[["영화명","month_start","polarity_mean"]],
        on=["영화명","month_start"], how="left"
    )
    panel_mean = ensure_gvar(panel_mean)

    # Save MEAN
    out_mean_csv = "movie_monthly_panel_emotions_mean.csv"
    out_mean_dta = "movie_monthly_panel_emotions_mean.dta"
    panel_mean.to_csv(out_mean_csv, index=False, encoding="utf-8-sig")
    panel_mean.to_stata(out_mean_dta, write_index=False, version=STATA_VERSION)
    coverage_summary(panel_mean, "OUTPUT: MEAN (P_* + polarity_mean)")

    # -------- Build MAX panel (M_* + polarity_mean) --------
    print("4) Build MAX (M_*) ...")
    df_max = compute_max_emotions(df)
    max_cols = [f"M_{e}" for e in PRIMARY_EMOTIONS]
    panel_max = create_monthly_panel(df_max, max_cols)

    print("4b) Attach monthly polarity_mean ...")
    panel_max = panel_max.merge(
        pol_panel[["영화명","month_start","polarity_mean"]],
        on=["영화명","month_start"], how="left"
    )
    panel_max = ensure_gvar(panel_max)

    # Save MAX
    out_max_csv = "movie_monthly_panel_emotions_max.csv"
    out_max_dta = "movie_monthly_panel_emotions_max.dta"
    panel_max.to_csv(out_max_csv, index=False, encoding="utf-8-sig")
    panel_max.to_stata(out_max_dta, write_index=False, version=STATA_VERSION)
    coverage_summary(panel_max, "OUTPUT: MAX (M_* + polarity_mean)")

    print("\n✨ All done.")

if __name__ == "__main__":
    main()


1) Load CSV ...
2) Parse & typing ...
3) Build MEAN (P_*) ...
3b) Attach monthly polarity_mean ...

[OUTPUT: MEAN (P_* + polarity_mean)] rows=12,738, movies=193, months=66
  review_count not-null cells: 3,481
  treated (gvar_month notna) cells: 12,738
4) Build MAX (M_*) ...
4b) Attach monthly polarity_mean ...

[OUTPUT: MAX (M_* + polarity_mean)] rows=12,738, movies=193, months=66
  review_count not-null cells: 3,481
  treated (gvar_month notna) cells: 12,738

✨ All done.
